# Sweep 11a — Labs Thesis: Core Modality Sweep

**Question:** Does adding labs help? How much compared to notes and copy?

**5 configs × 3 seeds = 15 runs ≈ 2–3 hours**

| Config | Modalities | Key question |
|--------|-----------|-------------|
| A0_naked | none | structural floor |
| A1_labs200 | + labs (200, flat) | clean Δ_labs |
| A2_labs200_copy | + labs + copy | copy amplifies lab signal? |
| A3_labs200_notes | + labs + notes (no copy) | notes + labs synergy? |
| A4_full | + labs + notes + copy | full system |

Run this first — A0 and A1 are the baseline and primary reference for all other sweeps (11b, 11c).

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q torch_geometric
!pip install -q pyyaml pandas numpy scikit-learn

In [ ]:
import os, sys, glob

if os.path.exists("/kaggle"):
    print("Running on Kaggle")
    os.chdir("/kaggle/working")
    os.system("rm -rf ./data ./src")
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/embeddings", exist_ok=True)

    train_paths = glob.glob("/kaggle/input/**/train.py", recursive=True)
    if not train_paths:
        raise FileNotFoundError("train.py not found in /kaggle/input")
    src_dir = os.path.dirname(train_paths[0])
    os.system(f"cp -r {src_dir} /kaggle/working/src")
    sys.path.append("/kaggle/working/src")

    for find_file in ["cohort_mimic3.pkl", "records_final.pkl"]:
        found = glob.glob(f"/kaggle/input/**/{find_file}", recursive=True)
        if found:
            processed_dir = os.path.dirname(found[0])
            print(f"Found processed dir at: {processed_dir}")
            for root, dirs, files in os.walk(processed_dir):
                rel = os.path.relpath(root, processed_dir)
                target_dir = os.path.join("./data/processed", rel) if rel != "." else "./data/processed"
                os.makedirs(target_dir, exist_ok=True)
                for fname in files:
                    src = os.path.join(root, fname)
                    link = os.path.join(target_dir, fname)
                    if not os.path.exists(link):
                        try:
                            os.symlink(src, link)
                        except OSError:
                            os.system(f"cp '{src}' '{link}'")
            break

    emb_paths = glob.glob("/kaggle/input/**/code_embeddings.pt", recursive=True)
    if not emb_paths:
        raise FileNotFoundError("code_embeddings.pt not found")
    emb_dir = os.path.dirname(emb_paths[0])
    for fpath in glob.glob(f"{emb_dir}/*"):
        fname = os.path.basename(fpath)
        link = f"./data/embeddings/{fname}"
        if not os.path.exists(link):
            os.symlink(fpath, link)

    reg_path = "/kaggle/working/src/model/registry.py"
    if os.path.exists(reg_path):
        with open(reg_path, "r") as rf:
            content = rf.read()
        if "import inspect" not in content:
            old = "        return cls(**kwargs)"
            new = (
                "        import inspect\n"
                "        sig = inspect.signature(cls.__init__)\n"
                "        has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD "
                "for p in sig.parameters.values())\n"
                "        if has_var_keyword:\n"
                "            filtered_kwargs = kwargs\n"
                "        else:\n"
                "            filtered_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}\n"
                "        return cls(**filtered_kwargs)"
            )
            if old in content:
                content = content.replace(old, new)
                with open(reg_path, "w") as wf:
                    wf.write(content)
                print("  registry.py patched")

print("Setup done:", os.getcwd())

In [ ]:
import yaml
from pathlib import Path

BASE_CONFIG = "src/config.yaml"

def make_config(output_path, model_overrides=None, preprocessing_overrides=None, smoke_epochs=None):
    with open(BASE_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    if model_overrides:
        for k, v in model_overrides.items():
            cfg["model"][k] = v
    if preprocessing_overrides:
        for k, v in preprocessing_overrides.items():
            cfg["preprocessing"][k] = v
    if smoke_epochs is not None:
        cfg["training"]["epochs"] = smoke_epochs
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return str(output_path)

# Shared backbone — same for all 11a/11b/11c configs
BACKBONE_MODEL = {
    "graph_encoder_type": "hgt_ehr_feat",
    "hgt_layers": 1,
    "pos_weight_cap": 15.0,
}
BACKBONE_FLAGS = (
    "--device cuda "
    "--visit_level_training "
    "--fusion_strategy film "
)

SEEDS = [42, 123, 456]
reports_dir = Path("experiment_reports/sweep_11a_modality")
results_log = []

print("Config helpers ready.")

In [ ]:
# Smoke test — 1 epoch per config before committing 3h
import subprocess
from pathlib import Path

Path("smoke_s11a").mkdir(exist_ok=True)

SMOKE = [
    {"name": "A0", "model_ov": {"copy_mechanism": False, **BACKBONE_MODEL},
     "prep_ov": {"note_method": "none", "lab_dim": 0}, "flags": ""},
    {"name": "A1", "model_ov": {"copy_mechanism": False, **BACKBONE_MODEL},
     "prep_ov": {"note_method": "none", "lab_dim": 400},
     "flags": "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200"},
    {"name": "A4", "model_ov": BACKBONE_MODEL,
     "prep_ov": {"lab_dim": 400},
     "flags": "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200"},
]

smoke_results = []
for t in SMOKE:
    cfg_path = f"s11a_smoke_{t['name']}.yaml"
    make_config(cfg_path, model_overrides=t["model_ov"],
                preprocessing_overrides=t["prep_ov"], smoke_epochs=1)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{BACKBONE_FLAGS} {t['flags']} "
           f"--seed 42 --results_dir smoke_s11a/{t['name']}")
    print(f"SMOKE {t['name']}: {cmd}")
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tail = (proc.stdout + proc.stderr).strip().split("\n")[-5:]
    for l in tail: print(l)
    status = "PASS" if proc.returncode == 0 else f"FAIL ({proc.returncode})"
    smoke_results.append(f"{status}: {t['name']}")
    print(f">>> {status}\n")

for r in smoke_results: print(r)
if any("FAIL" in r for r in smoke_results):
    raise RuntimeError("Smoke tests failed.")
print("All smoke tests passed.")

In [ ]:
import subprocess, gc, torch

GROUP_A = [
    {
        "name": "A0_naked",
        "model_ov": {"copy_mechanism": False, **BACKBONE_MODEL},
        "prep_ov": {"note_method": "none", "lab_dim": 0},
        "flags": "",
        "note": "Structural floor — LLM codes + graph only. No notes, no labs, no copy.",
    },
    {
        "name": "A1_labs200",
        "model_ov": {"copy_mechanism": False, **BACKBONE_MODEL},
        "prep_ov": {"note_method": "none", "lab_dim": 400},
        "flags": "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200",
        "note": "+ 200 labs, flat encoder. Clean Δ_labs = A1 - A0.",
    },
    {
        "name": "A2_labs200_copy",
        "model_ov": BACKBONE_MODEL,
        "prep_ov": {"note_method": "none", "lab_dim": 400},
        "flags": "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200",
        "note": "+ labs + copy (no notes). Does copy amplify lab signal?",
    },
    {
        "name": "A3_labs200_notes",
        "model_ov": {"copy_mechanism": False, **BACKBONE_MODEL},
        "prep_ov": {"lab_dim": 400},
        "flags": "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200",
        "note": "+ labs + notes (no copy). Notes+labs synergy?",
    },
    {
        "name": "A4_full",
        "model_ov": BACKBONE_MODEL,
        "prep_ov": {"lab_dim": 400},
        "flags": "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200",
        "note": "Full system: labs + notes + copy.",
    },
]

for cfg in GROUP_A:
    print(f"\n{'#'*70}\n# {cfg['name']} — {cfg['note']}\n{'#'*70}")
    cfg_path = f"s11a_{cfg['name']}.yaml"
    make_config(cfg_path, model_overrides=cfg["model_ov"], preprocessing_overrides=cfg["prep_ov"])

    for seed in SEEDS:
        run_name = f"{cfg['name']}_seed{seed}"
        run_dir  = reports_dir / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        log_path = run_dir / "training_log.txt"
        cmd = (
            f"python -u src/train.py --config {cfg_path} "
            f"{BACKBONE_FLAGS} {cfg['flags']} "
            f"--seed {seed} --results_dir {run_dir}"
        )
        print(f"\n=== {run_name} ===\n>> {cmd}\n")
        try:
            with open(log_path, "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_name}: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"\nDone — {len(results_log)} runs | {sum(1 for r in results_log if 'SUCCESS' in r)} succeeded")

In [ ]:
import json, numpy as np
from pathlib import Path

reports_dir = Path("experiment_reports/sweep_11a_modality")

def get_metric(d, key):
    sec = d.get("test_metrics", {})
    for k in [key, key.replace("_", " "), key.title()]:
        if k in sec: return float(sec[k])
        if k in d:   return float(d[k])
    return 0.0

results = {}
for jp in sorted(reports_dir.rglob("result_*.json")):
    with open(jp) as f: d = json.load(f)
    run_name = jp.parent.name
    idx = run_name.rfind("_seed")
    if idx == -1: continue
    results.setdefault(run_name[:idx], []).append(d)

def summarize(name):
    runs = results.get(name, [])
    if not runs: return None
    jac = [get_metric(d, "jaccard") for d in runs]
    f1  = [get_metric(d, "f1")      for d in runs]
    ddi = [get_metric(d, "ddi_rate") for d in runs]
    return {"jac_mean": np.mean(jac), "jac_std": np.std(jac, ddof=1) if len(jac)>1 else 0.0,
            "f1": np.mean(f1), "ddi": np.mean(ddi), "n": len(jac)}

print("\n" + "="*80)
print("SWEEP 11a — CORE MODALITY SWEEP RESULTS")
print("="*80)
print(f"  {'Config':<28} {'Jac mean±std':>16}  {'Δ naked':>8}  F1      DDI     n")
print(f"  {'-'*28} {'-'*16}  {'-'*8}  {'-'*6}  {'-'*6}  -")

a0_s = summarize("A0_naked")
a0   = a0_s["jac_mean"] if a0_s else 0.0

jacs = {}
for name, label in [
    ("A0_naked",         "A0: naked"),
    ("A1_labs200",       "A1: +200 labs"),
    ("A2_labs200_copy",  "A2: +200 labs+copy"),
    ("A3_labs200_notes", "A3: +200 labs+notes"),
    ("A4_full",          "A4: full system"),
]:
    s = summarize(name)
    if not s:
        print(f"  {label:<28} {'(missing)':>16}")
        jacs[name] = 0.0
        continue
    delta = f"{s['jac_mean']-a0:+.4f}" if name != "A0_naked" else "     —"
    print(f"  {label:<28} {s['jac_mean']:.4f}±{s['jac_std']:.4f}  {delta:>8}  {s['f1']:.4f}  {s['ddi']:.4f}  {s['n']}")
    jacs[name] = s["jac_mean"]

a0, a1, a2, a3, a4 = [jacs.get(k, 0) for k in ["A0_naked","A1_labs200","A2_labs200_copy","A3_labs200_notes","A4_full"]]
print(f"\n  Δ_labs        (A1−A0): {a1-a0:+.4f}")
print(f"  Δ_copy+labs   (A2−A0): {a2-a0:+.4f}  (copy on top of labs = {a2-a1:+.4f})")
print(f"  Δ_notes+labs  (A3−A0): {a3-a0:+.4f}  (notes on top of labs = {a3-a1:+.4f})")
print(f"  Δ_full        (A4−A0): {a4-a0:+.4f}")
superadd = (a4-a0) - ((a1-a0) + (a2-a1) + (a3-a1))
print(f"  Superadditivity:       {superadd:+.4f}  (+ = synergy, 0 = independent, − = redundant)")

In [ ]:
import zipfile
from pathlib import Path

zip_name = "reports_sweep_11a_modality.zip"
rd = Path("experiment_reports/sweep_11a_modality")
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(rd.rglob("result_*.json")): zf.write(p, p.relative_to(rd))
    for p in sorted(rd.rglob("training_log.txt")): zf.write(p, p.relative_to(rd))
print(f"Exported → {zip_name}")